# 04 — Sector Clustering
K-Means (K=5) over latest non-TTM financial metrics, visualized using PCA.

In [ ]:
from pathlib import Path
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
root=Path.cwd(); clean=root/'data'/'clean'; pl=pd.read_csv(clean/'profitandloss.csv'); bs=pd.read_csv(clean/'balancesheet.csv'); cf=pd.read_csv(clean/'cashflow.csv')
latest=lambda d:d.loc[~d.is_ttm.astype(bool)].sort_values('sort_order').groupby('symbol').tail(1)
frame=latest(pl)[['symbol','opm_pct','net_profit_margin_pct','interest_coverage']].merge(latest(bs)[['symbol','debt_to_equity']],on='symbol').merge(latest(cf)[['symbol','free_cash_flow','cash_conversion_ratio']],on='symbol')
features=frame.columns.drop('symbol'); x=StandardScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(frame[features])); frame['cluster']=KMeans(n_clusters=5,random_state=42,n_init=20).fit_predict(x); points=PCA(2,random_state=42).fit_transform(x); frame[['pc1','pc2']]=points
sns.scatterplot(data=frame,x='pc1',y='pc2',hue='cluster',palette='tab10'); plt.title('Company clusters (PCA projection)'); plt.show(); display(frame.groupby('cluster')[list(features)].mean().round(2)); frame.to_csv(root/'data'/'warehouse'/'sector_clusters.csv',index=False)